### DimUser

### Autoloader

In [0]:
df_user = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimUser/checkpoint")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/DimUser")        

In [0]:
display(df_user, checkpointLocation="abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimUser/display_checkpoint")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys

project_path = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_path)

from utils.transformations import resuable

In [0]:
df_user = df_user.withColumn("user_name", upper("user_name"))

In [0]:
import os
import sys

project_path = os.path.join(os.getcwd(), '..', '..')

sys.path.append(project_path)

from utils.transformations import resuable

In [0]:
df_user_obj = resuable()


In [0]:
df_user = df_user_obj.dropColumns(df_user[0] if isinstance(df_user, tuple) else df_user, ['_rescued_data'])

In [0]:
df_user = df_user.dropDuplicates(['user_id'])

## Writing the Data 

In [0]:
df_user.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimUser/checkpoint")\
        .trigger(once=True)\
        .option("path","abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimUser/data")\
        .toTable("spotify_cata.silver.DimUser")

In [0]:
%sql
select * from spotify_cata.silver.DimUser 

## DimArtist

In [0]:
df_art = spark.read.format("parquet")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/DimArtist")

In [0]:
display(df_art)

In [0]:
df_art = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimArtist/checkpoint_display")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/DimArtist")  

In [0]:
df_art_obj = resuable()

In [0]:
df_art = df_art_obj.dropColumns(df_art[0] if isinstance(df_art, tuple) else df_art, ['_rescued_data'])

In [0]:
df_art = df_art.dropDuplicates(['artist_id'])

In [0]:
df_art.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimArtist/checkpoint_write")\
        .trigger(once=True)\
        .option("path","abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimArtist/data")\
        .toTable("spotify_cata.silver.DimArtist")

In [0]:
%sql
select * from spotify_cata.silver.DimArtist

## DimTrack

In [0]:
df_track = spark.read.format("parquet")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/DimTrack")

In [0]:
display(df_track)

In [0]:
df_track = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimTrack/checkpoint_display")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/DimTrack")  

In [0]:
df_track = df_track.withColumn("durationflag", when(col("duration_sec")<150, "short")\
                                              .when(col("duration_sec")<300, "medium")\
                                              .otherwise("long"))

df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"),"-"," "))



In [0]:
df_track_obj = resuable()
df_track = df_track_obj.dropColumns(df_track, ["_rescued_data"])

In [0]:
df_track.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimTrack/checkpoint_write")\
        .trigger(once=True)\
        .option("path","abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimTrack/data")\
        .toTable("spotify_cata.silver.DimTrack")

In [0]:
%sql
select * from spotify_cata.silver.dimtrack

## DimDate

In [0]:
df_date = spark.read.format("parquet")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/DimDate")

In [0]:
display(df_date)

In [0]:
df_date = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimDate/checkpoint_display")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/DimDate") 

In [0]:
df_date_obj = resuable()
df_date = df_date_obj.dropColumns(df_date, ["_rescued_data"])


df_date = df_date.withColumn("weekday", upper("weekday"))

In [0]:
df_date.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimDate/checkpoint_write")\
        .trigger(once=True)\
        .option("path","abfss://silver@storagespotifymanuj.dfs.core.windows.net/DimDate/data")\
        .toTable("spotify_cata.silver.DimDate")

In [0]:
%sql
select * from spotify_cata.silver.dimdate

## FactStream

In [0]:
df_fact = spark.read.format("parquet")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/FactStream")

In [0]:
display(df_fact)

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@storagespotifymanuj.dfs.core.windows.net/FactStream/checkpoint_display")\
                .load("abfss://bronze@storagespotifymanuj.dfs.core.windows.net/FactStream") 

In [0]:
df_fact_obj = resuable()
df_fact = df_fact_obj.dropColumns(df_fact, ["_rescued_data"])

In [0]:
df_fact.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation","abfss://silver@storagespotifymanuj.dfs.core.windows.net/FactStream/checkpoint_write")\
        .trigger(once=True)\
        .option("path","abfss://silver@storagespotifymanuj.dfs.core.windows.net/FactStream/data")\
        .toTable("spotify_cata.silver.FactStream")

In [0]:
%sql
select * from spotify_cata.silver.factstream

In [0]:
%sql
select * from spotify_cata.gold.dimuser

In [0]:
%sql
select * from spotify_cata.gold.dimtrack
where track_id in(46,5)